# Data Reading from Silver Layer

# To See Dataframe Columns

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable


# 1. Load or bootstrap the Gold Dim & Silver For New Data

In [0]:
# df_new = spark.read.table('medallion_arc_e2e.silver.products')

# df_new = df_new.withColumn("updated_at", current_timestamp())

# from pyspark.sql.functions import round
# df_new = df_new.withColumn("discounted_price", round(col("discounted_price"), 2))


# Remove Duplicate

In [0]:
# df_new.groupBy(df.columns).count().filter(col('count') > 1).orderBy(col('product_id')).display()

In [0]:
# df_new = df_new.dropDuplicates(subset=['product_id'])

# display(df_new.limit(10))

In [0]:
# if spark.catalog.tableExists('medallion_arc_e2e.gold.dim_products'):
#     df_old = spark.read.table('medallion_arc_e2e.gold.dim_products')
# else:
#     df_old = spark.createDataFrame([], df_new.schema)

# 2. Join new vs old on business key

In [0]:
# df_join = df_new.alias("new").join(
#     df_old.filter(col("is_current") == True).alias("old"),  # ← only compare against current rows!
#     on="product_id",
#     how="left"
# )

In [0]:
# df_join.display()

# # df_join.filter(col('product_id') == 'P0015').display()

# 3. SCD Type 2 columns — changes that need history

In [0]:
# #       product_name, category, brand  → we VERSION these

# SCD2_CHANGED = (
#     (col("new.product_name") != col("old.product_name")) |
#     (col("new.category")     != col("old.category"))     |
#     (col("new.brand")        != col("old.brand"))
# )

# 4. SCD Type 1 columns — changes that just overwrite

In [0]:
# #       price, discounted_price, stock → we OVERWRITE these (no new row needed)

# SCD1_ONLY_CHANGED = (
#     ~SCD2_CHANGED &   # SCD2 columns did NOT change ...
#     (
#         (col("new.price")             != col("old.price")) |
#         (col("new.discounted_price")  != col("old.discounted_price"))
#     )
# )

# 5. Build the four record sets

## 5a. Completely NEW products (no old record exists)

In [0]:
# df_inserts = (
#     df_join.filter(col("old.product_id").isNull())
#     .select("new.*")
#     .withColumn("is_current",  lit(True))
#     .withColumn("start_date",  current_timestamp())
#     .withColumn("end_date",    lit(None).cast("timestamp"))
# )

In [0]:
# df_inserts.display()

## 5b. SCD2 change → EXPIRE the old row

In [0]:
# df_expired = (
#     df_join.filter(col("old.product_id").isNotNull() & SCD2_CHANGED)
#     .select("old.*")
#     .withColumn("is_current", lit(False))
#     .withColumn("end_date",   current_timestamp())   # close the old version
# )

In [0]:
# df_expired.display()

## 5c. SCD2 change → INSERT new version row

In [0]:
# df_new_versions = (
#     df_join.filter(col("old.product_id").isNotNull() & SCD2_CHANGED)
#     .select("new.*")
#     .withColumn("is_current", lit(True))
#     .withColumn("start_date", current_timestamp())
#     .withColumn("end_date",   lit(None).cast("timestamp"))
# )

In [0]:
# df_new_versions.display()

## 5d. SCD1-only change → UPDATE in place (no new row, just overwrite columns) We handle this via Delta MERGE below, not as a separate DataFrame

# 6. Unchanged rows — carry forward as-is

In [0]:
# df_unchanged = (
#     df_join.filter(
#         col("old.product_id").isNotNull() & ~SCD2_CHANGED & ~SCD1_ONLY_CHANGED
#     )
#     .select("old.*")
# )

In [0]:
# df_unchanged.display()

# 7. Union all rows that need to be written

In [0]:
# df_final = df_inserts \
#     .unionByName(df_expired) \
#     .unionByName(df_new_versions) \
#     .unionByName(df_unchanged)

In [0]:
# df_final.display()

# 8. Write using Delta MERGE (the correct SCD2 write pattern)

**MERGE is preferred over overwrite because:**
-   ✓ Atomically expires old rows AND inserts new ones
-   ✓ Handles SCD1 in-place updates cleanly
-   ✓ No risk of duplicating unchanged rows

In [0]:
# if spark.catalog.tableExists('medallion_arc_e2e.gold.dim_products'):
#     delta_table = DeltaTable.forName(spark, 'medallion_arc_e2e.gold.dim_products')

#     delta_table.alias("old").merge(
#         source = df_final.alias("new"),
#         condition = "old.product_id = new.product_id AND old.start_date = new.start_date"
#     ) \
#     .whenMatchedUpdate(                         # SCD1: overwrite price/stock in-place
#         condition = "old.is_current = true AND old.product_id = new.product_id",
#         set = {
#             "price":            "new.price",
#             "discounted_price": "new.discounted_price",
#             "is_current":       "new.is_current",
#             "end_date":         "new.end_date",
#             "updated_at":       "new.updated_at"
#         }
#     ) \
#     .whenNotMatchedInsertAll() \
#     .execute()

# else:
#     # First run — just create the table
#     df_final.write \
#         .format("delta") \
#         .mode("overwrite") \
#         .saveAsTable('medallion_arc_e2e.gold.dim_products')

In [0]:
%sql
-- Select * from medallion_arc_e2e.gold.dim_products
-- WHERE product_id = 'P0015';

In [0]:
%sql
-- TRUNCATE TABLE medallion_arc_e2e.gold.dim_products;

# Complete production-level SCD1 + SCD2 PySpark implementation using Delta MERGE without any DataFrame joins

![image_1775486845100.png](./image_1775486845100.png "image_1775486845100.png")

In [0]:
# ═══════════════════════════════════════════════════════════════════════════
#  SCD TYPE 1 + TYPE 2  |  Gold Dim Products
#  No DataFrame join — uses Delta MERGE directly against source
#  Two-MERGE pattern to avoid multiple-source-row conflict
# ═══════════════════════════════════════════════════════════════════════════

from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, when
)
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, BooleanType
from delta.tables import DeltaTable

TARGET_TABLE  = "medallion_arc_e2e.gold.dim_products"
SOURCE_TABLE  = "medallion_arc_e2e.silver.products"

# SCD2 → version history tracked  |  SCD1 → overwrite in-place
SCD2_COLS = ["product_name", "category", "brand"]
SCD1_COLS = ["price", "discounted_price", "stock"]
BUSINESS_KEY = "product_id"

In [0]:
df_s = spark.read.table('medallion_arc_e2e.silver.products')

df_s.orderBy('product_id').display()

In [0]:
# Duplicate Check

# df_s.groupBy(df_s.columns).count().filter(col('count') > 1).orderBy(col('product_id')).display()

In [0]:
# Remove Duplicate Using Window Function

# df_s = df_s.withColumn("rn", row_number().over(Window.partitionBy(df_s['product_id']).orderBy(col('ingesttime').desc()))).filter(col('rn') == 1).drop('rn')
# df_s.display()

In [0]:
# ── STEP 1: Load & deduplicate Silver ───────────────────────────────────────
# Never trust Silver to be clean — always deduplicate on business key first
# Most recent row wins (adjust order column to your data's recency signal)

dedup_window = Window.partitionBy(col('product_id')).orderBy(col("ingesttime").desc())

df_source = (
    spark.read.table('medallion_arc_e2e.silver.products')
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

# Hard stop — if dupes still exist something is very wrong upstream
dupe_count = df_source.groupBy(col('product_id')).count().filter(col("count") > 1).count()
assert dupe_count == 0, f"Duplicates found in source after dedup: {dupe_count} product_ids"

In [0]:
df_source.display()

In [0]:
# ── STEP 2: Create Gold table on first run ───────────────────────────────────
# Adds all SCD metadata columns so MERGE always has a fully-formed schema

if not spark.catalog.tableExists('medallion_arc_e2e.gold.dim_products'):
    (
        df_source.withColumn("is_current", lit(True).cast(BooleanType()))
        .withColumn("start_date", current_timestamp())
        .withColumn("end_date",   lit(None).cast(TimestampType()))
        .withColumn("updated_at", current_timestamp())
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable('medallion_arc_e2e.gold.dim_products')
    )
    print(f"[INIT] Gold table created: {TARGET_TABLE}")

else:

    # ── STEP 3: Register source as a temp view ───────────────────────────────
    # Delta MERGE works directly against registered views — no Python join needed
    # This is the key optimisation: Delta pushes predicates into the scan itself

    df_source.withColumn("updated_at", current_timestamp()) \
             .createOrReplaceTempView("products_source")


    # ── STEP 4: MERGE 1 — Expire SCD2 rows + Overwrite SCD1 columns ─────────
    #
    #  Merge condition: business_key + is_current = true
    #  → Guarantees exactly ONE target row matches each source row
    #  → No multiple-source-row conflict possible
    #
    #  WHEN MATCHED + SCD2 changed → expire old row (is_current=False)
    #  WHEN MATCHED + SCD1 changed → overwrite price/stock in-place
    #  WHEN MATCHED + nothing changed → Delta skips (no-op automatically)

    delta_table = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_table.alias("t")
        .merge(
            source    = spark.table("products_source").alias("s"),
            condition = f"t.{BUSINESS_KEY} = s.{BUSINESS_KEY} AND t.is_current = true"
        )

        # SCD2: name / category / brand changed → expire the current row
        .whenMatchedUpdate(
            condition = """
                t.product_name <> s.product_name OR
                t.category     <> s.category     OR
                t.brand        <> s.brand
            """,
            set = {
                "is_current": "false",
                "end_date":   "s.updated_at",
                "updated_at": "s.updated_at"
            }
        )

        # SCD1: only price / stock changed → overwrite in-place, no new row
        .whenMatchedUpdate(
            condition = """
                t.product_name  = s.product_name AND
                t.category      = s.category     AND
                t.brand         = s.brand         AND
                (
                    t.price            <> s.price            OR
                    t.discounted_price <> s.discounted_price
                )
            """,
            set = {
                "price":            "s.price",
                "discounted_price": "s.discounted_price",
                # "stock":            "s.stock",
                "updated_at":       "s.updated_at"
                # is_current / start_date / end_date stay untouched ✅
            }
        )
        .execute()
    )

    print("[MERGE 1] SCD2 rows expired + SCD1 rows updated in-place")

    # ── STEP 5: MERGE 2 — Insert new rows ────────────────────────────────────
    #
    #  After MERGE 1:
    #    → SCD2 products:  old row now has is_current=False  → no active row exists
    #    → New products:   no row exists at all
    #
    #  Merge condition: same business_key + is_current = true
    #  → Nothing matches for both cases above → falls into WHEN NOT MATCHED
    #  → Inserts new version row cleanly with no ambiguity ✅

    (
        delta_table.alias("t")
        .merge(
            source    = spark.table("products_source").alias("s"),
            condition = f"t.{BUSINESS_KEY} = s.{BUSINESS_KEY} AND t.is_current = true"
        )

        # Fires for: (a) brand new products  (b) SCD2 changed products (just expired above)
        .whenNotMatchedInsert(
            values = {
                "product_id":       "s.product_id",
                "product_name":     "s.product_name",
                "category":         "s.category",
                "brand":            "s.brand",
                "price":            "s.price",
                "discounted_price": "s.discounted_price",
                # "stock":            "s.stock",
                "is_current":       "true",
                "start_date":       "s.updated_at",
                "end_date":         "null",
                "updated_at":       "s.updated_at"
            }
        )
        .execute()
    )

    print("[MERGE 2] New products + new SCD2 versions inserted")

## 🔁 MERGE `.whenMatchedUpdate()` Evaluation Order (Important)

In Spark/Delta Lake, multiple `.whenMatchedUpdate()` clauses are evaluated **top to bottom**.

### ⚙️ How it works

For each matched row:

1. Spark checks the **first `.whenMatchedUpdate()` condition**
2. If the condition is ✅ TRUE:
   - The corresponding update is executed
   - 🚫 Remaining `.whenMatchedUpdate()` conditions are NOT checked
3. If the condition is ❌ FALSE:
   - Spark moves to the next `.whenMatchedUpdate()`

---

### 🧠 Key Rule

👉 **Only the FIRST matching condition is applied per row**

---

### 📌 Example (SCD Logic)

```python
.whenMatchedUpdate(  # 1️⃣ SCD Type 2
    condition = "business columns changed",
    set = {...}
)

.whenMatchedUpdate(  # 2️⃣ SCD Type 1
    condition = "only non-business columns changed",
    set = {...}
)

In [0]:
# ── STEP 6: Validate ─────────────────────────────────────────────────────────
# Run after every load — catches any logic bugs before they reach consumers

df_gold = spark.read.table(TARGET_TABLE)

# Rule 1: Each product_id must have exactly ONE active (is_current=True) row
active_dupes = (
    df_gold
    .filter(col("is_current") == True)
    .groupBy(BUSINESS_KEY)
    .count()
    .filter(col("count") > 1)
)
assert active_dupes.count() == 0, "VIOLATION: Multiple active rows for same product_id"

# Rule 2: All expired rows must have an end_date set
missing_end = (
    df_gold
    .filter((col("is_current") == False) & col("end_date").isNull())
)
assert missing_end.count() == 0, "VIOLATION: Expired row has NULL end_date"

# Rule 3: start_date must always be before end_date
bad_dates = (
    df_gold
    .filter(col("end_date").isNotNull() & (col("start_date") >= col("end_date")))
)
assert bad_dates.count() == 0, "VIOLATION: start_date >= end_date on expired row"

print("[VALIDATE] All checks passed ✅")
df_gold.groupBy("is_current").count().show()